# Run: tiny_cnn + FEMNIST, 20 rounds, WITH 7 active attackers (14%), checkpointed

Companion to `RUN_MVP_TINY_CNN_FEMNIST.ipynb` (the clean baseline run for
this same (model, dataset) case) -- same model, same data, same 50
clients, same 20 rounds, SAME checkpoint-resume machinery. The one
difference: 7 clients (14% -- matching
`ARCHITECTURE.md`'s own stated experimental target, contributions
table N7: "n=50 ... 7 adversaries") have their delta forced
every round to a model-replacement-scale malicious update (same
construction `RUN_ATTACKS.ipynb`'s F11 gate used to confirm V3 catches
this inside a real round), via `cheater_client_idx`/`cheater_force_delta`.
Everything else about each attacker's submission -- ring signature,
VRF, masking, sketching, the real range proof -- runs exactly as it
would for an honest client; see `fl/client.py`'s `_force_delta`
docstring. All 7 attackers reuse the SAME crafted delta (a
simplifying choice, noted as such -- not independently-drawn attacks
per client). Which clients cheat, and this notebook's own checkpoint
branch, both come from `fl/attacks/mvp_matrix.py` -- the single source
of truth across all 4 attack-variant notebooks (see that module's
docstring for why).

**Why 7, not some other number**: `ARCHITECTURE.md` SS2.2 (A2)
states the system's theoretical tolerance as `f < n/2 = 25` -- that's a
ceiling for a stress-test, not the number to validate against here.
N7's own stated target (7 of 50) is what makes this run
consistent with what the paper already commits to validating.

**Point of this run**: produce a SECOND accuracy-vs-round curve to put
next to the clean tiny_cnn+FEMNIST run -- "the other
43 clients converge normally for all 20 rounds while 7
attackers (14%) are rejected by V3 every single
round" is the strongest combined utility+security claim this paper can
make for this case, and it needs this run (not just F11's 3-round/
9-client gate) to back it at the real training scale.

**Expect more aborted rounds than the clean run.** Rejected clients
don't count toward `k_min` eligibility, so round-availability degrades
as the malicious fraction grows -- see `RUN_MVP_TINY_CNN_FEMNIST.ipynb`'s own
markdown for the exact abort-probability numbers at this config. More
ABORTED-round lines in this run's log than the clean run's is the
model working as predicted, not a new bug.

**Checkpoint durability: git, not Drive.** Checkpoints are committed
and pushed, after every round, to this notebook's own dedicated
`colab-checkpoints-case2-tiny_cnn-attack7` branch on the FL-ZKP repo itself -- distinct from
BOTH the clean tiny_cnn+FEMNIST run's branch AND every
other attack variant's branch (FIX.md F17: every MVP run notebook,
attack variants included, gets its own branch so concurrent pushes can
never collide), never `main` -- see the git-checkpoint-setup cell's own
markdown for the full reasoning. A push failing (transient network
blip) only costs a retry next round -- it does not stop training, and
the checkpoint is always safe on local disk regardless.

**If Colab disconnects partway through:** re-run from cell 1 again.
The git-checkpoint-setup cell re-clones the dedicated checkpoint
checkout fresh and fetches whatever was last pushed to
`colab-checkpoints-case2-tiny_cnn-attack7` -- the run cell then finds this notebook's own
checkpoint (by filename, see the eval markdown below) and resumes
after the last completed round, it does not restart from round 0.


In [ ]:
# 1. Clone/pull the repo. (No Drive mount -- checkpoints go through
# git instead, see the git-checkpoint-setup cell further down and this
# notebook's own intro markdown for why.)
import os

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP


In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121


**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).


In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())


In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)


In [ ]:
# 5. Load FEMNIST, writer-disjoint non-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.femnist.cache_dir points at a
# Drive path this run's Drive doesn't have mounted -- overridden here
# to a local Colab cache dir instead, same no-Drive pattern the MNIST
# run uses for its "source" override. flwr-datasets downloads and
# caches FEMNIST (~200MB) to this path over the network, no Drive
# dependency at all.
from fl.data_noniid import prepare_case2

case2_data = prepare_case2(seed=CFG['seed'], cache_dir="/content/femnist_cache/")
print("clients:", len(case2_data['client_train']))
print("server_root writers:", len(case2_data['split'].server_root_writers))
print("unseen_test writers (headline metric source):", len(case2_data['split'].unseen_test_writers))


In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


In [ ]:
# 7. Build the model.
# ~20.6K params, matches WhiteCloak/PQS-BFL's model scale (see fl/models.py build_tiny_cnn docstring) -- same backbone as the MNIST attack run, different (non-IID, writer-split) dataset.
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=62, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## Checkpoint durability via git (not Drive)

**Why a separate clone, a separate branch, and a callback -- not just
`git commit` inside `/content/FL-ZKP`:**
- A SEPARATE clone (`/content/FL-ZKP-checkpoints`) so committing a
  checkpoint can never touch the code checkout this notebook is actually
  importing from -- no risk of a `git checkout`/`reset` on the
  checkpoint side disturbing `sys.path`-imported modules mid-run.
- A dedicated branch (shown as `CKPT_BRANCH` in the cell below), never
  `main` -- small binary commits (one per round, re-run disconnects
  add more) have no business sitting in the same history a reviewer
  sees when they clone this repo to read the paper's actual code.
  **Each of the 4 MVP run notebooks gets its OWN branch** (FIX.md F17,
  2026-11-17) -- an earlier version shared one branch across all four,
  keyed apart only by checkpoint filename, and every single push
  collided when they ran concurrently (observed live: 12/12 rounds
  failed), even with a fetch+rebase retry (F16). A dedicated ref per
  notebook removes the race at the git level instead of mitigating it.
- `fl/round_loop.py`'s `on_checkpoint_saved` hook (2026-11-16) keeps
  the FL library itself git-agnostic -- it just calls back with
  `(checkpoint_path, round_idx)` after every local save; all the git
  mechanics below live here, in the notebook, not in the library.

**You'll need a GitHub Personal Access Token** with push access to
`YashasweeNarayan/FL-ZKP` (classic PAT, `repo` scope, or a fine-grained
token scoped to just this repo). The cell below prompts for it via
`getpass` (never echoed to the cell's output, never written to a file
or committed) and deletes it from memory immediately after use.

**On a disconnect and re-run**: this clone lives on local Colab disk,
so it does NOT survive the disconnect (same as the code clone) -- but
everything pushed to this notebook's own `CKPT_BRANCH` before the
disconnect DID survive, on GitHub. Re-running this cell re-clones
fresh and fetches that branch if it already exists, so the run cell's
`resume_from` still finds the latest checkpoint and continues from
there, not round 0.


In [ ]:
# 7.5. Git-backed checkpoint durability (replaces Drive -- see markdown above).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/kaggle/working/FL-ZKP-checkpoints"
# 2026-11-17: each of the 4 MVP run notebooks gets its OWN branch, not
# one shared branch -- see FIX.md F17. Running all four at once on one
# shared branch caused every single push to collide and fail (12/12
# rounds, observed live) even with F16's fetch+rebase retry, because
# the retry itself can lose the SAME race under real concurrent load.
# A dedicated ref per notebook removes the race at the git level
# instead of mitigating it after the fact.
CKPT_BRANCH = "colab-checkpoints-case2-tiny_cnn-attack7"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    # does this notebook's own CKPT_BRANCH already exist on the remote
    # (pushed in an earlier, since-disconnected session)? if so, use it --
    # that's where the actual latest checkpoint lives. If not, this is new.
    # explicit destination refspec -- a plain "fetch origin <branch>"
    # only updates FETCH_HEAD, not refs/remotes/origin/<branch>, so the
    # checkout below would fail even though the fetch itself succeeded.
    # Verified against a real throwaway git remote before using this.
    _fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
    if _fetch.returncode == 0:
        _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
        print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
    else:
        _git(["checkout", "-b", CKPT_BRANCH])
        print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh.")
    # keep the remote URL with the embedded token only as long as needed
    # for this session's pushes; drop the plaintext token variable now.
    del _token
else:
    print("checkpoint checkout already present this session -- reusing it.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """on_checkpoint_saved callback for run_training_loop (fl/round_loop.py,
    2026-11-16). Never raises -- a push failure costs a retry next round,
    not the training run; the checkpoint is already safe on local disk
    regardless of whether this push succeeds.

    2026-11-17 (FIX.md F17): each MVP run notebook now has its OWN
    dedicated CKPT_BRANCH, so this retry is defense-in-depth, not the
    primary fix -- it used to be the ONLY mitigation for 4 notebooks
    sharing one branch, which collided on every single push under real
    concurrent load (observed live). Kept in case this exact branch is
    ever reused by two sessions at once (e.g. the same notebook run
    twice): fetch+rebase onto the remote tip and retry once before
    giving up for this round (next round tries again regardless)."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            # Lost a race with a concurrent writer on this branch --
            # fetch + rebase onto the remote tip and retry once.
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# 8. The real run: 20 rounds, checkpointed every round and pushed to
# this notebook's own dedicated 'colab-checkpoints-case2-tiny_cnn-attack7' git branch (see the
# git-checkpoint-setup cell above -- CKPT_DIR/push_checkpoint_to_git
# come from there, not Drive) -- WITH 7 active attackers
# (14%, matching ARCHITECTURE.md's own N7 target) the
# whole time.
#
# Same construction F11 used to confirm V3 catches A1 inside a real
# round (FIX.md F11): each attacker's delta is forced every round to
# attacks.a1_scaling.craft_scaled_update's gamma*(w_target - w_t) output,
# built fresh from THIS model's actual initial weights (all 7
# share the SAME crafted delta -- a simplifying choice, not
# independently-drawn attacks per client). Everything else about each
# attacker's submission (ring signature, VRF, masking, sketching, the
# real range proof) runs exactly as it would for an honest client --
# see fl/client.py's `_force_delta` docstring for why this is NOT the
# same as a lying/forged submission. _cheat_zero_sketch does NOT also
# fire here (fl/round_loop.py's F11 fix: it's conditional on
# cheater_force_delta being None), so this is a clean, single-mechanism
# attack across all 7, not the composed one F11 found and fixed.
import numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint
from attacks.a1_scaling import craft_scaled_update
from fl.attacks.mvp_matrix import CHEATER_INDICES

REAL_ROUNDS = 20
CHEATER_CLIENT_IDX = list(CHEATER_INDICES)  # single source of truth: fl/attacks/mvp_matrix.py

# Own checkpoint filename -- distinct from the clean run's and from
# every other attack variant's, so this run never cross-resumes.
CKPT_PATH = f'{CKPT_DIR}/case2_{REAL_MODEL}_r{REAL_ROUNDS}_withattack7.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"bootstrap-{REAL_MODEL}-case2-withattack7".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

# crafted ONCE, from this model's actual initial weights -- same
# gamma*(w_target - w_t) construction as the F11 gate, far enough from a
# random target that it should never pass V3 regardless of which round
# it's submitted in. All 7 attackers submit this SAME delta --
# see markdown above for why that's a noted simplification.
w_target = np.random.default_rng(CFG['seed'] + 99).normal(0, 1.0, size=dim)
# F18 follow-up (2026-11-17): flat_w lives on whatever device model_cell
# moved the model to (F15, 2026-11-16) -- .cpu() first or this raises
# "can't convert cuda:0 device type tensor to numpy" on any real GPU
# runtime. craft_scaled_update itself is pure numpy, CPU-only by design.
malicious_delta = craft_scaled_update(flat_w.cpu().numpy().astype(np.float64), w_target)
print(f"crafted malicious delta (clients {CHEATER_CLIENT_IDX}), raw norm:", np.linalg.norm(malicious_delta))

print("="*70)
print(f"MALICIOUS CLIENTS IN THIS RUN: {CHEATER_CLIENT_IDX} ({len(CHEATER_CLIENT_IDX)} of {n_clients} = {100*len(CHEATER_CLIENT_IDX)/n_clients:.0f}%)")
print("="*70)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case2_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    cheater_client_idx=CHEATER_CLIENT_IDX,
    cheater_force_delta=malicious_delta,
    on_checkpoint_saved=push_checkpoint_to_git,
)
release_gpu_memory()
n_accepted = sum(1 for h in real_history if not h['aborted'])
print(f"done: {len(real_history)} rounds in history, {n_accepted} accepted (not aborted)")

print("\nper-round attacker verdict:")
attacker_ever_accepted = {}
attacker_ever_rejected = {}
for h in real_history:
    if h.get("aborted"):
        print(f"  round {h.get('round')}: ABORTED -- {h.get('reason')}")
        continue
    reasons = h.get("rejected_reasons", {})
    accepted_idxs = set(h.get("accepted_client_indices", []))
    rejected_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in reasons)
    accepted_this_round = sorted(i for i in CHEATER_CLIENT_IDX if i in accepted_idxs)
    for i in rejected_this_round:
        attacker_ever_rejected[i] = True
    for i in accepted_this_round:
        attacker_ever_accepted[i] = True
    print(f"  round {h.get('round')}: accepted={h.get('accepted')}, "
          f"attackers rejected={rejected_this_round}, attackers accepted={accepted_this_round}, "
          f"aggregate_exact={h.get('aggregate_exact')}")

if attacker_ever_accepted:
    print(f"\nFAIL: attacker(s) {sorted(attacker_ever_accepted)} had their scaled update accepted at least once -- a real defense failure, report this.")
elif attacker_ever_rejected:
    print(f"\nPASS: every attacker that was ever eligible ({sorted(attacker_ever_rejected)}) was rejected by V3 every time, "
          f"across all {REAL_ROUNDS} rounds, while the honest majority kept training normally.")
else:
    print("\nINCONCLUSIVE: no attacker was ever sampled/eligible in any round -- check sampling_rate/eligibility.")


## Evaluate accuracy

Works whether you let training finish, stopped it early (Ctrl-C /
interrupt), or are coming back in a fresh runtime with nothing but the
checkpoint pushed to this notebook's own `CKPT_BRANCH`: it loads the LATEST checkpointed weights directly
from `CKPT_PATH` (falling back to the in-memory `final_w_t` only if no
checkpoint file exists yet), so it always reports accuracy for
whatever round you actually reached, not a partially-updated `model`
object.


In [ ]:
import torch
from fl.models import unflatten_to_state_dict

def evaluate_accuracy(model, images, labels, batch_size=512):
    # GPU fix (2026-11-16): dataset tensors stay on CPU by design (see
    # fl/client.py's 2026-11-16 comment); move each eval batch to
    # wherever `model` actually lives, same pattern as training.
    device = next(model.parameters()).device
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch_images = images[i:i + batch_size].to(device)
            batch_labels = labels[i:i + batch_size].to(device)
            logits = model(batch_images)
            preds = logits.argmax(dim=1)
            correct += (preds == batch_labels).sum().item()
    return correct / len(images)

# Load the LATEST checkpointed weights explicitly -- see markdown above.
ckpt = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
eval_w_t = ckpt['w_t'] if ckpt is not None else final_w_t
eval_round = ckpt['round'] if ckpt is not None else 'final (in-memory, no checkpoint found)'
model.load_state_dict(unflatten_to_state_dict(eval_w_t.to(next(model.parameters()).device), model_shapes))

# unseen_test: writers held out entirely from every client/server_root/
# global_val role (ROADMAP.md §3.4) -- THE headline number for FEMNIST,
# since it's the only split that tests generalization to writers the
# federation never saw during training at all, non-IID handwriting
# style included.
unseen_ds = case2_data['unseen_test']
unseen_acc = evaluate_accuracy(model, unseen_ds.images, unseen_ds.labels)

# global_val: held-out SEEN-writer samples (disjoint per-client holdout,
# not disjoint writers) -- secondary sanity check, not the headline
# number, same role as MNIST's global_val.
val_ds = case2_data['global_val']
val_acc = evaluate_accuracy(model, val_ds.images, val_ds.labels)

print(f"{REAL_MODEL} @ round {eval_round}: unseen-writer FEMNIST test accuracy = {unseen_acc:.4f} ({len(unseen_ds.images)} images)")
print(f"{REAL_MODEL} @ round {eval_round}: held-out global_val accuracy = {val_acc:.4f} ({len(val_ds.images)} images)")
